In [2]:
from pathlib import Path
from collections import defaultdict
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/Users/gupta/Documents/DIS-IND/data/wikipedia"
CHUNK_SIZE = 200_000


# ============================================================
# ANALYZER
# ============================================================

def analyze_dataset(dataset_path, chunk_size=100_000):
    dataset_path = Path(dataset_path)

    csv_files = sorted(dataset_path.glob("*.csv"))
    tbl_files = sorted(dataset_path.glob("*.tbl"))

    if csv_files and tbl_files:
        raise ValueError(
            "Dataset contains both .csv and .tbl files. "
            "Use only one format per dataset."
        )

    if csv_files:
        files = csv_files
        file_type = "csv"
        # The downloaded Sawfish CENSUS CSV files are semicolon-separated.
        separator = ";"
    elif tbl_files:
        files = tbl_files
        file_type = "tbl"
        separator = "|"
    else:
        raise FileNotFoundError(f"No .csv or .tbl files found in {dataset_path}")

    total_rows = 0
    max_rows_per_table = 0
    total_attributes = 0
    all_attribute_distinct_counts = []
    all_dataset_values = set()
    value_to_attributes = defaultdict(set)
    table_reports = []

    total_size_bytes = sum(file.stat().st_size for file in files)
    total_size_mb = total_size_bytes / (1024 * 1024)

    for file_path in files:
        print(f"Analyzing: {file_path.name}")
        table_rows = 0
        table_size_mb = file_path.stat().st_size / (1024 * 1024)

        if file_type == "csv":
            header = pd.read_csv(file_path, sep=separator, nrows=0)
            columns = list(header.columns)
        else:
            sample = pd.read_csv(
                file_path, sep=separator, header=None, nrows=1, dtype=str
            )
            if len(sample.columns) > 0 and sample.iloc[:, -1].isna().all():
                number_of_columns = len(sample.columns) - 1
            else:
                number_of_columns = len(sample.columns)
            columns = [f"column_{i + 1}" for i in range(number_of_columns)]

        number_of_attributes = len(columns)
        total_attributes += number_of_attributes
        distinct_values = {column: set() for column in columns}

        if file_type == "csv":
            reader = pd.read_csv(
                file_path,
                sep=separator,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False,
            )
        else:
            reader = pd.read_csv(
                file_path,
                sep=separator,
                header=None,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False,
            )

        for chunk in reader:
            if file_type == "tbl" and len(chunk.columns) > len(columns):
                chunk = chunk.iloc[:, :len(columns)]

            chunk.columns = columns
            table_rows += len(chunk)

            for column in columns:
                unique_values = chunk[column].unique()
                distinct_values[column].update(unique_values)
                all_dataset_values.update(unique_values)
                qualified_attribute = f"{file_path.stem}.{column}"
                for value in unique_values:
                    value_to_attributes[value].add(qualified_attribute)

        total_rows += table_rows
        max_rows_per_table = max(max_rows_per_table, table_rows)
        table_distinct_counts = []

        for column in columns:
            count = len(distinct_values[column])
            table_distinct_counts.append(count)
            all_attribute_distinct_counts.append(count)

        table_max_distinct = max(table_distinct_counts, default=0)
        table_avg_distinct = (
            sum(table_distinct_counts) / len(table_distinct_counts)
            if table_distinct_counts
            else 0
        )
        table_reports.append(
            {
                "table": file_path.name,
                "size_mb": table_size_mb,
                "rows": table_rows,
                "attributes": number_of_attributes,
                "max_distinct": table_max_distinct,
                "average_distinct": table_avg_distinct,
            }
        )

    max_distinct_values_per_attribute = max(
        all_attribute_distinct_counts, default=0
    )
    average_distinct_values_per_attribute = (
        sum(all_attribute_distinct_counts) / len(all_attribute_distinct_counts)
        if all_attribute_distinct_counts
        else 0
    )
    total_distinct_values_dataset = len(all_dataset_values)
    number_of_clusters = len(
        {frozenset(attributes) for attributes in value_to_attributes.values()}
    )

    print()
    print("=" * 80)
    print("DATASET SUMMARY")
    print("=" * 80)
    print(f"Number of tables                       : {len(files):,}")
    print(f"Dataset size (MB)                      : {total_size_mb:,.2f}")
    print(f"Total rows                             : {total_rows:,}")
    print(f"Max # rows per table                   : {max_rows_per_table:,}")
    print(f"Total # attributes                     : {total_attributes:,}")
    print(
        "Max # distinct values per attribute    : "
        f"{max_distinct_values_per_attribute:,}"
    )
    print(
        "Average # distinct values per attribute: "
        f"{average_distinct_values_per_attribute:,.2f}"
    )
    print(
        "Total distinct values in dataset       : "
        f"{total_distinct_values_dataset:,}"
    )
    print(f"Total # clusters                       : {number_of_clusters:,}")

    print()
    print("=" * 80)
    print("TABLE SUMMARY")
    print("=" * 80)
    for table in table_reports:
        print(
            f"{table['table']:<35} "
            f"Rows={table['rows']:<12,} "
            f"Attributes={table['attributes']:<6,} "
            f"Size={table['size_mb']:>10,.2f} MB "
            f"MaxDistinct={table['max_distinct']:>12,} "
            f"AvgDistinct={table['average_distinct']:>12,.2f}"
        )

    return {
        "number_of_tables": len(files),
        "dataset_size_mb": total_size_mb,
        "total_rows": total_rows,
        "max_rows_per_table": max_rows_per_table,
        "total_attributes": total_attributes,
        "max_distinct_values_per_attribute": max_distinct_values_per_attribute,
        "average_distinct_values_per_attribute": average_distinct_values_per_attribute,
        "total_distinct_values_dataset": total_distinct_values_dataset,
        "number_of_clusters": number_of_clusters,
        "tables": table_reports,
    }


report = analyze_dataset(DATASET_PATH, chunk_size=CHUNK_SIZE)


Analyzing: WIKIPEDIA.csv

DATASET SUMMARY
Number of tables                       : 1
Dataset size (MB)                      : 586.73
Total rows                             : 14,024,428
Max # rows per table                   : 14,024,428
Total # attributes                     : 11
Max # distinct values per attribute    : 5,475,188
Average # distinct values per attribute: 649,330.09
Total distinct values in dataset       : 6,950,344
Total # clusters                       : 32

TABLE SUMMARY
WIKIPEDIA.csv                       Rows=14,024,428   Attributes=11     Size=    586.73 MB MaxDistinct=   5,475,188 AvgDistinct=  649,330.09


In [1]:
import pandas as pd
from pathlib import Path

project_dir = Path("../")

input_file = project_dir / "data/wikipedia/wikipedia/WIKIPEDIA.csv"
output_dir = project_dir / "data/wikipedia"
output_dir.mkdir(exist_ok=True)

# Number of rows to generate
target_rows = [1_000_000,2_000_000,4_000_000,6_000_000,8_000_000]

# Load only up to the maximum required rows
df = pd.read_csv(input_file, nrows=max(target_rows))

print(f"Loaded {len(df):,} rows")

for n in target_rows:
    if len(df) < n:
        print(f"Skipping {n:,}: input contains only {len(df):,} rows")
        continue

    output_file = output_dir / f"{input_file.stem}_{n}.csv"

    df.iloc[:n].to_csv(output_file, index=False)

    print(f"Created {output_file} with {n:,} rows")

Loaded 8,000,000 rows
Created ../data/wikipedia/WIKIPEDIA_1000000.csv with 1,000,000 rows
Created ../data/wikipedia/WIKIPEDIA_2000000.csv with 2,000,000 rows
Created ../data/wikipedia/WIKIPEDIA_4000000.csv with 4,000,000 rows
Created ../data/wikipedia/WIKIPEDIA_6000000.csv with 6,000,000 rows
Created ../data/wikipedia/WIKIPEDIA_8000000.csv with 8,000,000 rows


In [2]:
import pandas as pd
from pathlib import Path

project_dir = Path("../")

input_file = project_dir / "data/wikipedia/wikipedia/WIKIPEDIA.csv"
output_dir = project_dir / "data/wikipedia"
output_dir.mkdir(parents=True, exist_ok=True)

# Fixed number of rows for all column-scalability datasets
fixed_rows = 1_000_000

# Increasing numbers of columns
target_cols = [5, 10]

# Load fixed number of rows
df = pd.read_csv(input_file, sep=";",nrows=fixed_rows)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

for n in target_cols:
    if len(df.columns) < n:
        print(
            f"Skipping {n} columns: "
            f"input contains only {len(df.columns)} columns"
        )
        continue

    # Keep the same rows and take the first n columns
    subset = df.iloc[:, :n]

    output_file = output_dir / f"{input_file.stem}_{n}cols.csv"

    subset.to_csv(output_file, index=False)

    print(
        f"Created {output_file} "
        f"with {len(subset):,} rows × {n} columns"
    )

/var/folders/cl/x0sfq21n3m75zvtn65kr_53w0000gn/T/ipykernel_95813/273446348.py:17: DtypeWarning: Columns (0: IMAGE.csv - column6, 1: IMAGE.csv - column7, 2: IMAGE.csv - column8, 3: IMAGE.csv - column12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, sep=";",nrows=fixed_rows)


Loaded 1,000,000 rows and 11 columns
Created ../data/wikipedia/WIKIPEDIA_5cols.csv with 1,000,000 rows × 5 columns
Created ../data/wikipedia/WIKIPEDIA_10cols.csv with 1,000,000 rows × 10 columns
